---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 2.3 — Judge Alignment with Human Feedback

---

## 💬 Product Team Check-in:

> **Sam:** "The metrics from the custom judges are looking good, but we have an increasing amount of examples where the judge thinks a response is ok but the ML teams would say that it doesn't pass. Is there a way to go back and adjust the judge based on their specific feedback?"


**Objective:** Use MLflow's **judge alignment** workflow to teach a judge to match human preferences by learning from expert feedback on real traces.

In this notebook:

1. Generate traces from our MLflow assistant
2. Run an initial judge on those traces
3. Provide human feedback (simulated ground truth)
4. Align the judge using `judge.align()` with SIMBA optimizer
5. Compare before/after accuracy

---
## 0 — Connect to MLflow

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-1", alias="EXPERIMENT_1_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    model: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_model: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"


    ##ALIGN JUDGE MODEL TO CONFIG
    #Both the judge's assessments and human feedback must share the same assessment name 
    # on the same traces for alignment to work.
    align_judge: str = "response_quality"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
from openai import OpenAI
import mlflow

openai_client = OpenAI(
    api_key=cfg.gemini_api_key.get_secret_value(),
    base_url=cfg.gemini_openai_base_url,
)

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.experiment_name)
mlflow.openai.autolog()

## 1 — Generate traces from the assistant

We'll run a set of questions through our MLflow assistant and collect the traces. These represent real production-like interactions that we want to evaluate.

In [ ]:
SYSTEM_PROMPT = mlflow.genai.load_prompt(cfg.prompt_alias)

# Questions that cover a range of difficulty and topics
test_questions = [
    "How do I log a model with mlflow?",
    "What is the difference between mlflow.start_run() and mlflow.start_span()?",
    "How do I set up autologging for PyTorch?",
    "Can MLflow track GPU metrics during training?",
    "How do I deploy a model from the MLflow registry to a REST endpoint?",
    "What is MLflow Tracing and when should I use it?",
    "How do I compare two runs in the MLflow UI?",
    "What's the best way to log a pandas DataFrame as an artifact?",
    "How do I use mlflow.evaluate() for LLM evaluation?",
    "Can I use MLflow with Weights & Biases at the same time?",
    "How do I configure a remote tracking server?",
    "What is the MLflow Deployments API?",
    "How do I search for runs with specific metrics above a threshold?",
    "What scorer should I use to check if responses are grounded in retrieved context?",
    "How do I version prompts in MLflow 3?",
]

In [ ]:
trace_ids = []

for q in test_questions:
    response = openai_client.chat.completions.create(
        model=cfg.model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.template},
            {"role": "user", "content": q},
        ],
        temperature=cfg.temperature,
        max_completion_tokens=cfg.max_tokens,
    )
    # autolog captures the trace automatically
    trace_ids.append(mlflow.get_last_active_trace_id())
    print(f"  ✓ {q[:60]}..." if len(q) > 60 else f"  ✓ {q}")

print(f"\nCollected {len(trace_ids)} trace IDs")

In [ ]:
trace_ids

---
## 2 — Create the initial judge

We define a custom judge using `make_judge()` that evaluates whether the assistant's response is **helpful and accurate** for an MLflow question. This is our *unaligned* judge — it uses generic instructions.

In [ ]:
from mlflow.genai.judges import make_judge
from typing import Literal

response_quality_judge = make_judge(
    name=cfg.align_judge,
    instructions=(
        "Evaluate whether the assistant's response is helpful and accurate "
        "for the given MLflow question.\n\n"
        "User question: {{ inputs }}\n"
        "Assistant response: {{ outputs }}\n\n"
        "Rate as: good, acceptable, or poor."
    ),
    feedback_value_type=Literal["good", "acceptable", "poor"],
    model=cfg.judge_model,
)

print(f"Judge created: {cfg.align_judge}")
print(f"  Feedback values: good | acceptable | poor")

### Run the judge on all traces

The judge evaluates each trace and logs its assessment automatically. We'll also peek at what the judge thinks to see where it might disagree with a human reviewer.

In [ ]:
judge_results = []

for i, trace_id in enumerate(trace_ids):
    trace = mlflow.get_trace(trace_id)
    inputs = trace.data.spans[0].inputs
    outputs = trace.data.spans[0].outputs

    result = response_quality_judge(inputs=inputs, outputs=outputs)
    judge_results.append(result)
    print(f"  [{result.value:>10}]  {test_questions[i][:65]}")

print(f"\nJudge assessed all {len(judge_results)} traces")

---
## 3 — Provide human feedback

This is where the magic happens. In production you'd collect this through the MLflow UI's **"Add Feedback"** button, or from a labeling team. Here we simulate expert review.

Notice: the human sometimes **disagrees** with the judge — that's the whole point. The expert has domain-specific standards (e.g., they want code examples, or they penalize outdated API references).

> **Critical:** The feedback `name` must match the judge's name (`"response_quality"`) so the alignment optimizer can pair them.

In [ ]:
# Simulated expert feedback — the human reviewer has stricter standards
# than the generic judge. They want:
#   - Code examples for "how do I" questions
#   - Accurate MLflow 3 APIs (not deprecated ones)
#   - Honest "I don't know" over hallucinated answers

human_labels = [
    "good",        # log a model — straightforward, expects code
    "acceptable",  # start_run vs start_span — nuanced, partial credit
    "good",        # autologging PyTorch — well-documented topic
    "poor",        # GPU metrics — likely hallucinated, MLflow doesn't do this natively
    "acceptable",  # deploy from registry — complex topic, partial answer is fine
    "good",        # tracing overview — core MLflow 3 feature
    "good",        # compare runs — basic UI feature
    "acceptable",  # log DataFrame — multiple valid approaches
    "good",        # mlflow.evaluate() — should nail this
    "poor",        # MLflow + W&B — not a real integration, should say so
    "acceptable",  # remote tracking server — config-heavy, partial is fine
    "poor",        # Deployments API — renamed/restructured in MLflow 3
    "good",        # search runs — well-documented
    "good",        # RetrievalGroundedness scorer — specific, correct answer expected
    "good",        # version prompts — new MLflow 3 feature
]

assert len(human_labels) == len(trace_ids), "Need one label per trace"
print(f"Human labels ready: {len(human_labels)} reviews")

In [ ]:
# Log human feedback on each trace
for trace_id, label, question in zip(trace_ids, human_labels, test_questions):
    mlflow.log_feedback(
        trace_id=trace_id,
        name=ALIGN_JUDGE_NAME,  # must match judge name
        value=label,
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="ml_engineer_review",
        ),
    )

print(f"Logged {len(human_labels)} human feedback assessments")
print("Each trace now has both a judge assessment AND a human label.")

### Quick comparison: judge vs. human

Let's see where the judge and the human expert disagree — these disagreements are what the alignment optimizer will learn from.

In [ ]:
disagreements = 0
for i, (jr, hl) in enumerate(zip(judge_results, human_labels)):
    match = "✓" if jr.value == hl else "✗ DISAGREE"
    if jr.value != hl:
        disagreements += 1
    print(f"  {match:>12}  judge={jr.value:<10}  human={hl:<10}  {test_questions[i][:55]}")

agreement_rate = (len(judge_results) - disagreements) / len(judge_results) * 100
print(f"\nAgreement rate: {agreement_rate:.0f}% ({len(judge_results) - disagreements}/{len(judge_results)})")
print(f"Disagreements: {disagreements} — these are what alignment will fix.")

---
## 4 — Align the judge

Now we run the **SIMBA alignment optimizer**. It analyzes the disagreements between the judge and the human expert, then optimizes the judge's prompt to better match human preferences.

Under the hood, SIMBA uses DSPy's simplified multi-bootstrap aggregation to refine the judge's instructions.

In [ ]:
# Add dspy to the project dependencies and sync
uv add dspy
uv sync

In [ ]:
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id

# Retrieve all traces (they now have both judge + human assessments)
# traces_for_alignment = mlflow.search_traces(
#     experiment_ids=[experiment_id],
#     max_results=20,
#     return_type="list",
# )

# For simplicity
traces_for_alignment = [mlflow.get_trace(trace_id) for trace_id in trace_ids]

print(f"Retrieved {len(traces_for_alignment)} traces for alignment")
print(f"Minimum required: 10")

In [ ]:
# Run alignment — this calls the LLM to optimize the judge's prompt
optimizer = SIMBAAlignmentOptimizer(model="gemini:/gemini-3-flash-preview")

aligned_judge = response_quality_judge.align(
    traces_for_alignment,
    optimizer,
)

print("Judge aligned successfully!")
print(f"\nOriginal instructions:\n  {response_quality_judge.instructions[:120]}...")
print(f"\nAligned instructions:\n  {aligned_judge.instructions[:120]}...")

---
## 5 — Validate alignment improvement

Let's re-run the aligned judge on the same traces and check if its ratings now match the human expert more closely.

In [ ]:
aligned_results = []

for trace_id in trace_ids:
    trace = mlflow.get_trace(trace_id)
    inputs = trace.data.spans[0].inputs
    outputs = trace.data.spans[0].outputs
    result = aligned_judge(inputs=inputs, outputs=outputs)
    aligned_results.append(result)

# Compare original vs aligned
print(f"{'Question':<55}  {'Human':<10}  {'Original':<10}  {'Aligned':<10}")
print("-" * 95)

original_correct = 0
aligned_correct = 0

for i, (orig, aligned, human) in enumerate(zip(judge_results, aligned_results, human_labels)):
    o_match = orig.value == human
    a_match = aligned.value == human
    original_correct += o_match
    aligned_correct += a_match

    flag = ""
    if not o_match and a_match:
        flag = " ← fixed"
    elif o_match and not a_match:
        flag = " ← regressed"

    print(f"  {test_questions[i][:53]:<55} {human:<10} {orig.value:<10} {aligned.value:<10}{flag}")

print(f"\n{'Metric':<30} {'Before':>10} {'After':>10}")
print("-" * 52)
print(f"{'Agreement with human':<30} {original_correct:>10}/{len(trace_ids)} {aligned_correct:>10}/{len(trace_ids)}")
print(f"{'Accuracy':<30} {original_correct/len(trace_ids)*100:>9.0f}% {aligned_correct/len(trace_ids)*100:>9.0f}%")

---
## 6 — Register the aligned judge

Once we're satisfied with the improvement, we register the aligned judge so it can be reused in future evaluations across the team.

In [ ]:
aligned_judge.register(experiment_id=experiment_id)

print("Aligned judge registered to experiment.")
print("It can now be used in mlflow.genai.evaluate() like any other scorer.")

In [ ]:
experiment_id

In [ ]:
response_quality = mlflow.genai.get_scorer(ALIGN_JUDGE_NAME, experiment_id=experiment_id)

In [ ]:
# Load evaluation dataset
evaluation = mlflow.genai.evaluate(
    dataset=eval_dataset,
    judges=[response_quality],
)

---
## Recap

| Step | What we did | Key API |
|---|---|---|
| Generate traces | Ran the assistant on 15 questions | `client.chat.completions.create()` |
| Initial judge | Created a generic quality judge | `make_judge()` |
| Human feedback | Logged expert labels on each trace | `mlflow.log_feedback()` |
| Alignment | Optimized judge to match human standards | `judge.align()` + `SIMBAAlignmentOptimizer` |
| Validation | Compared accuracy before/after | Re-ran judge, measured agreement |
| Registration | Saved aligned judge for reuse | `aligned_judge.register()` |

**The alignment lifecycle is iterative** — as your assistant evolves and your quality bar shifts, you collect fresh feedback and re-align. The judge grows with your product.